# TRINETRA: first Siamese U-Net
Use a free GPU runtime. Only Kuro Siwo events are used. Trishuli and EMSR927 are excluded. Run cells in order. Proper training starts with fresh weights after the real-data sanity check. The first GPU run uses a bounded event-balanced subset, not the entire dataset.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU first.'
print(torch.__version__, torch.cuda.get_device_name(0))

Upload `TRINETRA_Siamese_UNet.zip` from the assistant response. The package contains code, split indices, two real samples and the completed sanity checkpoint/report. No credentials are included.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archives = [n for n in uploaded if n.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one project ZIP.'
ROOT = Path('/content/trinetra')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve()), 'Unsafe archive path'
    archive.extractall(ROOT)
os.chdir(ROOT)
print(ROOT)

In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_*training*.py', '-v')
run('scripts/train_siamese.py', '--mode', 'sanity', '--steps', '40')

Optional: run the next cell to save checkpoints to your Drive. Otherwise set OUTPUT to a local /content path and download outputs before the runtime expires.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/TRINETRA/siamese_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)

Fetch only an event-balanced subset: 16 patches for each of 27 train events and 8 for each of 6 validation events. No test imagery, no full shards. The hard network cap is 1 GiB. If a quota cannot be found within that budget, the command stops and records the reason; training must wait for a complete subset.

In [ ]:
run('scripts/fetch_kuro_training.py', '--train-per-event', '16', '--val-per-event', '8', '--max-mib', '1024')

Start proper GPU training. Best checkpoint selection uses validation macro-event IoU; threshold is fixed at 0.5. All invalid pixels are excluded. Stop after up to 20 epochs or 5 epochs without improvement.

In [ ]:
run('scripts/train_siamese.py', '--mode', 'fit', '--epochs', '20', '--batch-size', '8', '--out', str(OUTPUT))

After an interruption, rerun setup/data verification and resume with the same configuration, data and output directory. Do not execute this cell after an already completed run.

In [ ]:
# Uncomment only to resume an interrupted run:
# run('scripts/train_siamese.py', '--mode', 'fit', '--epochs', '20', '--batch-size', '8', '--out', str(OUTPUT), '--resume', str(OUTPUT/'last.pt'))

In [ ]:
import json
history = json.loads((OUTPUT/'history.json').read_text())
print(json.dumps(history[-1], indent=2))
print('Checkpoints:', OUTPUT/'best.pt', OUTPUT/'last.pt')

Sanity scores are training-patch overfit diagnostics. GPU validation scores assess held-out events in the fetched subset. Test and Trishuli/EMSR927 have not been used. Keep the deterministic fallback available; the Kuro Siwo GRD/Trishuli Gamma0 domain difference must be resolved before operational inference.